# L80 · 向量相似度检索 — 余弦相似度 vs 点积 vs L2，纯 NumPy k-NN 实现

**目标**：用 L2 归一化 embedding 做余弦相似度搜索，找最相似的 k 首歌。

🔗 **Aurora 连接**：本节实现的 `find_similar()` 将成为 `aurora.music.recommend` 的底层检索引擎，Month 5 接入 FAISS 时只需替换这个函数。

每首歌经过 encoder 后变成一个高维向量（embedding）。两首歌风格越接近，它们的向量方向越一致——用余弦相似度量这种方向接近程度，再用 top-k 选出最相似的歌，就构成了最简洁的检索系统。

In [ ]:
import numpy as np
import time

## 1. 余弦相似度与单位向量

余弦相似度定义：

```
cos_sim(a, b) = dot(a, b) / (||a|| * ||b||)
```

值域 `[-1, 1]`：1 表示方向完全相同（最相似），-1 表示方向相反，0 表示正交（无关）。

**关键技巧**：如果先把所有向量 L2 归一化（变成单位向量，||v|| = 1），则 `cos_sim(a, b) = dot(a, b)`，矩阵乘法一步搞定，不需要每次除以模长。

In [ ]:
# 演示：L2 归一化后点积 == 余弦相似度
a = np.array([3.0, 4.0])   # 模长 5
b = np.array([1.0, 0.0])   # 模长 1

# 原始余弦相似度
cos_raw = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 归一化后点积
a_n = a / np.linalg.norm(a)
b_n = b / np.linalg.norm(b)
cos_normed = np.dot(a_n, b_n)

print(f'原始余弦相似度: {cos_raw:.4f}')
print(f'归一化后点积:   {cos_normed:.4f}')
print(f'相等: {np.isclose(cos_raw, cos_normed)}')

## 2. 暴力 top-k：argpartition vs argsort

从 N 个分数中找最大的 k 个，有两种做法：

```
argsort:        O(N log N)  — 完整排序，返回全部排名
argpartition:   O(N)        — 只保证 top-k 进入最后 k 个槽，内部不排序
```

当 N 很大（百万首歌）而 k 很小（返回 5 首）时，`argpartition` 明显更快。用法：`np.argpartition(scores, -k)[-k:]` 拿到 top-k 的索引（顺序随机），再用 `scores[idx].argsort()[::-1]` 对这 k 个排序。

In [ ]:
# 演示：argpartition vs argsort 速度对比
rng = np.random.default_rng(42)
N, k = 100_000, 5
scores = rng.random(N)

t0 = time.perf_counter()
for _ in range(200):
    idx_sort = np.argsort(scores)[-k:][::-1]
t_sort = (time.perf_counter() - t0) / 200 * 1e6

t0 = time.perf_counter()
for _ in range(200):
    idx_part = np.argpartition(scores, -k)[-k:]
    idx_part = idx_part[np.argsort(scores[idx_part])[::-1]]
t_part = (time.perf_counter() - t0) / 200 * 1e6

print(f'argsort      平均: {t_sort:.1f} µs')
print(f'argpartition 平均: {t_part:.1f} µs')
print(f'结果一致: {np.array_equal(idx_sort, idx_part)}')

## 3. FAISS：大规模近似最近邻

[FAISS](https://github.com/facebookresearch/faiss) 是 Meta 开发的向量检索库，支持 IVF、HNSW 等索引结构，在百万量级下仍能毫秒级返回结果。

本节先用暴力 `np.dot` 实现，接口固定为 `(indices, scores)`。Month 5 只需把函数体换成 FAISS `index.search()`，上层代码零改动。

```
暴力检索（本节）:  适合 N < 10 万，无需安装额外依赖
FAISS（Month 5）:  适合 N > 10 万，需 pip install faiss-cpu
```

In [ ]:
# 演示：FAISS 接口预览（Month 5 才真正安装，这里只看调用形态）
print('暴力检索接口:')
print('  indices, scores = find_similar(query_emb, library_embs, top_k=5)')
print()
print('FAISS 接口（Month 5）:')
print('  import faiss')
print('  index = faiss.IndexFlatIP(d)  # 内积 = 余弦（已归一化）')
print('  index.add(library_embs)')
print('  scores, indices = index.search(query_emb[None], top_k)')

## 4. ✏️ 实现 `find_similar(query_emb, library_embs, top_k=5)`

**推理路线**：
1. L2 归一化 `query_emb`：`q = query_emb / ||query_emb||`
2. L2 归一化 `library_embs` 的每一行：`L = library_embs / ||library_embs||（按行）`
3. `scores = L @ q`，shape `(N,)`，每个元素是一首歌与 query 的余弦相似度
4. `top_k_idx = np.argpartition(scores, -top_k)[-top_k:]`，O(N) 拿到候选
5. 对候选按 scores 降序排列，返回 `(indices, scores[indices])`

**参考输入输出**：
```
query  = [1, 0, 0, 0]   # 爵士曲 embedding（单位向量）
library = np.eye(4)      # 4 首歌，各自方向正交
find_similar(query, library, top_k=2)
# => indices=[0, 1 或 2 或 3], scores=[1.0, 0.0]
# index 0 得分 1.0（完全匹配），其余得分 0.0
```

<details><summary>点击查看参考实现</summary>

```python
def find_similar(
    query_emb: np.ndarray,
    library_embs: np.ndarray,
    top_k: int = 5,
) -> tuple[np.ndarray, np.ndarray]:
    # L2 归一化
    q = query_emb / (np.linalg.norm(query_emb) + 1e-12)
    norms = np.linalg.norm(library_embs, axis=1, keepdims=True) + 1e-12
    L = library_embs / norms
    # 余弦相似度 = 内积（归一化后）
    scores = L @ q
    # top-k via argpartition
    top_k = min(top_k, len(scores))
    part_idx = np.argpartition(scores, -top_k)[-top_k:]
    # 对 top-k 内部排序（降序）
    order = np.argsort(scores[part_idx])[::-1]
    indices = part_idx[order]
    return indices, scores[indices]
```

</details>

In [ ]:
def find_similar(
    query_emb: np.ndarray,
    library_embs: np.ndarray,
    top_k: int = 5,
) -> tuple[np.ndarray, np.ndarray]:
    """余弦相似度 top-k 检索。

    参数
    ----
    query_emb   : shape (d,)   查询向量
    library_embs: shape (N, d) 歌曲库 embedding
    top_k       : 返回最相似的 k 首

    返回
    ----
    indices : shape (top_k,)  在 library_embs 中的行索引，按相似度降序
    scores  : shape (top_k,)  对应余弦相似度
    """
    # ✏️ TODO 1: L2 归一化 query_emb -> q
    q = ...

    # ✏️ TODO 2: L2 归一化 library_embs 每行 -> L
    L = ...

    # ✏️ TODO 3: scores = L @ q，shape (N,)
    scores = ...

    # ✏️ TODO 4: argpartition 取 top_k 候选，再降序排列
    top_k = min(top_k, len(scores))
    part_idx = ...
    order = ...
    indices = ...

    return indices, scores[indices]

In [ ]:
# 基础正确性检查
embs = np.eye(4)  # 4 首歌，各自正交
idx, s = find_similar(embs[0], embs, top_k=2)

assert 0 in idx, f'index 0（完全匹配）必须在 top-2 中，得到 {idx}'
assert np.isclose(s[0], 1.0), f'最高分应为 1.0，得到 {s[0]:.4f}'
assert s[0] >= s[1], '分数应降序排列'
print('✅ 基础检查通过')

# 返回形状检查
rng = np.random.default_rng(0)
big_lib = rng.standard_normal((100, 32))
q = rng.standard_normal(32)
idx2, s2 = find_similar(q, big_lib, top_k=10)
assert idx2.shape == (10,), f'indices shape 应为 (10,)，得到 {idx2.shape}'
assert s2.shape == (10,), f'scores shape 应为 (10,)，得到 {s2.shape}'
assert all(s2[i] >= s2[i+1] for i in range(9)), '分数未降序'
print('✅ 形状与降序检查通过')

## 5. 参数实验：50首歌库，查询10次

**参数**：`N=50`（歌曲数），`d=128`（embedding 维度），`top_k=5`

**预期现象**：
- 每次检索时间 < 1ms（暴力 dot 在 N=50 量级极快）
- `scores` 范围在 `[-1, 1]`，最高分接近 1（若 query 来自库中）
- 调大 `top_k` 到 50 结果仍正确（等于返回全部歌曲排名）

In [ ]:
rng = np.random.default_rng(7)
N, d, top_k = 50, 128, 5
library = rng.standard_normal((N, d))

times = []
for i in range(10):
    query = library[i]  # 用库中第 i 首歌作为 query，期望自身得分最高
    t0 = time.perf_counter()
    idx, scores = find_similar(query, library, top_k=top_k)
    elapsed_us = (time.perf_counter() - t0) * 1e6
    times.append(elapsed_us)
    top1_is_self = idx[0] == i
    print(f'query={i:2d}  top1={idx[0]:2d}(自身={top1_is_self})  '
          f'score={scores[0]:.4f}  时间={elapsed_us:.1f}µs')

avg_ms = np.mean(times) / 1000
print(f'\n平均检索时间: {avg_ms:.3f} ms  (应 < 1 ms for N={N})')
assert avg_ms < 1.0, f'检索太慢：{avg_ms:.3f} ms'

## 小结

`find_similar()` 接受一个查询 embedding 和歌曲库矩阵，输出按余弦相似度降序排列的 `(indices, scores)` 二元组。它将作为 `aurora.music.recommend` 的底层检索引擎，上层只需传入 encoder 生成的 embedding，无需了解检索细节。Month 5 引入 FAISS 后，只需替换此函数内部实现，接口保持不变——下一节 MU4 将在此基础上构建完整的播放列表推荐流水线。